# Introduction to Options

**Options** are financial contracts that give the holder the right, but not the obligation, to buy or sell an underlying asset (such as a stock) at a specified price (the **strike price**) on or before a certain date (the **expiration date**).

Options are called **derivatives** because their value is derived from the price of another asset (the underlying asset). They are widely used for hedging, speculation, and income generation in financial markets.

## Types of Options

- **Call Option:** Gives the holder the right to **buy** the underlying asset at the strike price.
- **Put Option:** Gives the holder the right to **sell** the underlying asset at the strike price.



## Payoff Diagrams

The payoff of an option at expiration depends on the relationship between the underlying asset price ($S$) and the strike price ($K$):

### Call Option Payoff

$$
\text{Payoff}_{\text{call}} = \max(S - K, 0)
$$


### Put Option Payoff

$$
\text{Payoff}_{\text{put}} = \max(K - S, 0)
$$


- For a **call option**, the payoff increases as the underlying price rises above the strike price.
- For a **put option**, the payoff increases as the underlying price falls below the strike price.

Options provide flexible strategies for managing risk and taking positions on future price movements in financial markets.

The code below generates payoff diagrams

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
K = 100  # Strike price
S = np.linspace(50, 150, 200)  # Range of underlying prices at expiry

# Payoff calculations
call_payoff = np.maximum(S - K, 0)
put_payoff = np.maximum(K - S, 0)

# Plot Call Option Payoff
plt.figure(figsize=(8, 4))
plt.plot(S, call_payoff, label='Call Option Payoff', color='blue')
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(K, color='gray', linestyle='--', label='Strike Price (K)')
plt.title('Call Option Payoff at Expiry')
plt.xlabel('Stock Price at Expiry ($S$)')
plt.ylabel('Payoff')
plt.legend()
plt.grid(True)
plt.show()

# Plot Put Option Payoff
plt.figure(figsize=(8, 4))
plt.plot(S, put_payoff, label='Put Option Payoff', color='red')
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(K, color='gray', linestyle='--', label='Strike Price (K)')
plt.title('Put Option Payoff at Expiry')
plt.xlabel('Stock Price at Expiry ($S$)')
plt.ylabel('Payoff')
plt.legend()
plt.grid(True)
plt.show()

## The Black-Scholes Formula

The **Black-Scholes formula** is a fundamental result in financial mathematics used to calculate the theoretical price of European call and put options, which are options that can only be exercised at expiration. It assumes that the price of the underlying asset follows a Geometric Brownian Motion with constant drift and volatility.

### Black-Scholes Formula for a European Call Option

$$
C = S_0 N(\frac{\ln(S_0/K) + (r + \frac{1}{2}\sigma^2)T}{\sigma \sqrt{T}}) - K e^{-rT} N(\frac{\ln(S_0/K) + (r + \frac{1}{2}\sigma^2)T}{\sigma \sqrt{T}} - \sigma \sqrt{T})
$$

where:
- $ C $ = price of the European call option
- $ S_0 $ = current price of the underlying asset
- $ K $ = strike price
- $ r $ = risk-free interest rate (annualized)
- $ T $ = time to maturity (in years)
- $ N(\cdot) $ = cumulative distribution function of the standard normal distribution

<br><br>

The formula can be written in a slightly more readable and understandable form as:

$$
C = S_0 N(d_1) - K e^{-rT} N(d_2)
$$

The terms $ d_1 $ and $ d_2 $ are defined as:

$$
d_1 = \frac{\ln(S_0/K) + (r + \frac{1}{2}\sigma^2)T}{\sigma \sqrt{T}}
$$
$$
d_2 = d_1 - \sigma \sqrt{T}
$$

where:
- $ \sigma $ = volatility of the underlying asset (annualized)

### Explanation

- The Black-Scholes formula provides a **closed-form solution** for pricing European options, which can only be exercised at expiration.
- The formula assumes no dividends, constant volatility, and a frictionless market.
- $ N(d_1) $ and $ N(d_2) $ represent the probabilities (under the risk-neutral measure) that the option will finish in the money.
- The term $ S_0 N(d_1) $ represents the expected value of receiving the stock at expiration, while $ K e^{-rT} N(d_2) $ is the present value of paying the strike price.

This model is widely used in financial markets for option pricing and risk management. 

*We will cover more about GBM in the next notebook.*

In [ ]:
# Simple implementation of the Black-Scholes formula for European options.

import math
from scipy.stats import norm
def black_scholes(S, K, T, r, sigma, option_type='call'):
    """
    Calculate Black-Scholes option price.
    
    Parameters:
    S: Current stock price
    K: Strike price
    T: Time to expiration (years)
    r: Risk-free rate (annual)
    sigma: Volatility (annual standard deviation)
    option_type: 'call' or 'put'
    """
    
    # Step 1: Calculate d1
    # d1 measures how far the stock price is from the strike price,
    # adjusted for drift and volatility over time
    # 
    # Breaking down the formula:
    # - log(S/K): Log return from current price to strike (moneyness)
    # - (r + 0.5 * sigma**2) * T: Expected drift with volatility adjustment
    # - sigma * sqrt(T): Standard deviation over time period T
    # 
    # d1 represents the number of standard deviations the stock price
    # is expected to move, normalized by volatility
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    
    # Step 2: Calculate d2
    # d2 = d1 - sigma * sqrt(T)
    # d2 adjusts d1 by subtracting one standard deviation
    # This accounts for the risk-neutral probability
    # d2 is used to calculate the probability the option expires in-the-money
    d2 = d1 - sigma * math.sqrt(T)
    
    # Step 3: Calculate option price based on type
    if option_type == 'call':
        # Call option formula: S * N(d1) - K * e^(-rT) * N(d2)
        # 
        # S * N(d1): Expected value of stock if option is exercised
        #   - N(d1) is the delta (hedge ratio)
        #   - Represents present value of receiving stock if option ends ITM
        # 
        # K * e^(-rT) * N(d2): Present value of paying strike price
        #   - e^(-rT): Discount factor for strike price
        #   - N(d2): Risk-neutral probability of option expiring ITM
        #   - Represents present value of paying strike if exercised
        price = S * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)
    else:
        # Put option formula: K * e^(-rT) * N(-d2) - S * N(-d1)
        # 
        # K * e^(-rT) * N(-d2): Present value of receiving strike price
        #   - N(-d2): Risk-neutral probability put expires ITM
        # 
        # S * N(-d1): Present value of delivering stock
        #   - N(-d1): Probability of delivering stock (negative delta)
        price = K * math.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    
    return price

### Sample implementation

In [ ]:

# Parameters
S = 100      # Stock price
K = 100      # Strike price
T = 1        # 1 year to expiration
r = 0.05     # 5% risk-free rate
sigma = 0.2  # 20% volatility

# Calculate prices
call_price = black_scholes(S, K, T, r, sigma, 'call')
put_price = black_scholes(S, K, T, r, sigma, 'put')

print(f"Call Option Price: ${call_price:.2f}")
print(f"Put Option Price: ${put_price:.2f}")